# Data gathering script
This program aims to gather data relevant for ML model training and summoner performance score calculation.

In [1]:
import sys
sys.path.insert(0, '/Users/mateuszpilawski/Desktop/github/riotdev/src')
from pipeline import load_api_key, pipeline

import pandas as pd
import numpy as np

In [2]:
api_key = load_api_key()
test_obj = pipeline(api_key=api_key, platform='EUW1', player_name='401dmg', player_tag='6969', count=1, save=False)
players = test_obj[0]['players']
game_duration = test_obj[0]['metadata']['gameDuration_min']

Caller puuid fetched 0.67
Matches ID fetched 0.79
Matches raw data fetched 0.98
Matches raw data fetched 0.98
Match object number 1 created. 1.50658
objects created sucessfuly :) 
 Timer: 1.51
Match object number 1 created. 1.50658
objects created sucessfuly :) 
 Timer: 1.51


### Data cleanup
in this step, we drop unecessary data preserved in server response. This includes player personal data and irrelevant statistics

In [3]:
for player in players:
    to_drop = ['puuid', 'username', 'championImageLink', 'summonerLevel', 'skillshotsDodged', 'visionWardsBoughtInGame', 'skillshotsHit', 'soloKills', 'doubleKills', 'tripleKills', 'quadraKills', 'pentaKills', 'killingSprees', 'largestKillingSpree', 'itemsPurchased', 'needVisionPings', 'enemyVisionPings', 'allInPings', 'pushPings', 'assistMePings', 'commandPings', 'dangerPings', 'enemyMissingPings', 'onMyWayPings', 'retreatPings', 'physicalDamageDealtToChampions', 'magicDamageDealtToChampions', 'trueDamageDealtToChampions', 'physicalDamageTaken', 'totalHealsOnTeammates', 'totalDamageShieldedOnTeammates', 'summoners', 'runes', 'items', 'icon', 'caller', 'masteries', 'visionScore', 'wardsPlaced',	'wardsKilled', 'detectorWardsPlaced', 'timeCCingOthers']
    for n in to_drop:
        player.pop(n, None)

    to_drop_meta = ['leagueId', 'queueType', 'leaguePoints']
    for i in to_drop_meta:
        player['metadata'].pop(i, None)

    string = ''
    for n in player['metadata'].values():
        string += str(n) + ' '
    player['Meta'] = string
    player.pop('metadata')

In [4]:
df = pd.DataFrame(players)

df['win'] = df['win'].map(lambda x: int(x == True))
df['gameEndedInSurrender'] = df['gameEndedInSurrender'].map(lambda x: int(x == True))

df['game_duration'] = game_duration

cols_with_per = ['teamDamagePercentage', 'damageTakenOnTeamPercentage', 'Meta']    
for col in cols_with_per:
    df[col] = df[col].str.replace('%', '')

In [13]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
to_normalize = [
    "kills",
    "deaths",
    "assists",
    "KDA",
    "killParticipation",
    "teamDamagePercentage",
    "damageTakenOnTeamPercentage",
    "damagePerMinute",
    "goldPerMinute",
    "visionScorePerMinute",
    "maxCsAdvantageOnLaneOpponent",
    "goldEarned",
    "totalMinionsKilled",
    "cs_min",
    "totalDamageDealtToChampions",
    "totalDamageTaken",
    "damageSelfMitigated",
    "damageDealtToObjectives",
    "damageDealtToTurrets",
    "totalTimeSpentDead"
]

for col in to_normalize:    
    df[col] = scaler.fit_transform(df[[col]]).flatten().round(3)

In [14]:
df.to_csv('test1.csv', sep=',')